# Fully Bayesian SAAS GP

Use SAAS when the input dimension is moderately high and only a small subset of dimensions is expected to matter. This notebook shows both single-task and multi-task wrappers. Fully Bayesian fitting uses NUTS rather than `make_mll()`.

> This example intentionally uses small MCMC settings so it remains practical as documentation. Increase `warmup_steps` and `num_samples` for real analyses.

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.fit import fit_fully_bayesian_model_nuts
from robotorchan.models import SaasFullyBayesianSingleTaskGP, SaasFullyBayesianMultiTaskGP

torch.manual_seed(0)
dtype = torch.double

## 1. High-dimensional synthetic data

In [ ]:
n, d = 28, 12
train_X = torch.rand(n, d, dtype=dtype)
def f(X):
    return (torch.sin(2 * torch.pi * X[..., 0]) + 0.8 * (X[..., 2] - 0.5) ** 2 - 0.6 * X[..., 5]).unsqueeze(-1)
train_Y = f(train_X) + 0.03 * torch.randn(n, 1, dtype=dtype)
train_X.shape, train_Y.shape

## 2. Single-task SAAS

In [ ]:
model = SaasFullyBayesianSingleTaskGP(train_X, train_Y)
print(model.raw_train_X.shape, model.raw_train_Y.shape)
print('supports_mll =', model.supports_mll)

In [ ]:
fit_fully_bayesian_model_nuts(
    model,
    warmup_steps=32,
    num_samples=16,
    thinning=2,
    disable_progbar=True,
)

In [ ]:
x0 = torch.linspace(0, 1, 120, dtype=dtype)
X_test = torch.full((120, d), 0.5, dtype=dtype)
X_test[:, 0] = x0
with torch.no_grad():
    posterior = model.posterior(X_test)
mean = posterior.mean.squeeze(-1)
std = posterior.variance.sqrt().squeeze(-1)
plt.figure(figsize=(7, 4))
plt.plot(x0, mean)
plt.fill_between(x0, mean - 2 * std, mean + 2 * std, alpha=0.2)
plt.xlabel('x0')
plt.ylabel('posterior')
plt.title('SAAS posterior slice')
plt.show()

## 3. Multi-task SAAS

In [ ]:
x_mt = torch.rand(18, d, dtype=dtype)
X0 = torch.cat([x_mt, torch.zeros(18, 1, dtype=dtype)], dim=-1)
X1 = torch.cat([x_mt, torch.ones(18, 1, dtype=dtype)], dim=-1)
Y0 = f(x_mt)
Y1 = 0.7 * f(x_mt) + 0.2
train_X_mt = torch.cat([X0, X1], dim=0)
train_Y_mt = torch.cat([Y0, Y1], dim=0)
mt_model = SaasFullyBayesianMultiTaskGP(train_X_mt, train_Y_mt, task_feature=-1)
print(mt_model.raw_train_X.shape, mt_model.raw_train_Y.shape)
print('supports_mll =', mt_model.supports_mll)

In [ ]:
fit_fully_bayesian_model_nuts(
    mt_model,
    warmup_steps=24,
    num_samples=12,
    thinning=2,
    disable_progbar=True,
)

## 4. When to use / not use

- Use SAAS when dimensionality is high and sparse relevance is plausible.
- Prefer standard or MAP-SAAS models when runtime is important.
- `supports_mll=False` is intentional: NUTS is the fitting route.
- Do not place this notebook in a fast PR CI path without reducing or skipping MCMC execution.